In [1]:
import torch
import torch.nn as nn
from torch import tensor
import torch.nn.functional as F

### Using Pytorch

In [63]:
attention = nn.MultiheadAttention(
    embed_dim=2,
    num_heads=1,
    batch_first=True,
    bias=False
)

attention.load_state_dict({
    "in_proj_weight": torch.tensor([
        # W_Q
        [0.1, 0.2],
        [0.3, 0.4],

        # W_K
        [0.2, 0.9],
        [0.4, 0.5],

        # W_V
        [0.3, 0.4],
        [0.5, 0.6],
    ]),
    
    "out_proj.weight": torch.tensor([
        [1.0, 1.1],
        [1.2, 1.3],
    ])
})

x = torch.tensor([
    [
        [0.2, 0.7],
        [0.4, 0.5],
    ]
])

output, weights = attention(
    query=x,
    key=x,
    value=x
)
lin = nn.Linear(2, 1, bias=False)
lin.load_state_dict({
    "weight": torch.tensor([
        [0.1, 0.2]
    ])
})

y_pred = lin(output[0][-1])
y_target = tensor([0.5])

loss_fn = nn.MSELoss()

loss = loss_fn(y_pred, y_target)
loss.backward()

In [64]:
lin.weight.grad

tensor([[-0.3548, -0.4216]])

In [65]:
attention.in_proj_weight.grad, attention.out_proj.weight.grad

(tensor([[-5.5953e-05, -6.9941e-05],
         [-7.9932e-06, -9.9915e-06],
         [ 2.7977e-05, -2.7977e-05],
         [ 6.3946e-05, -6.3946e-05],
         [-4.0479e-02, -8.1331e-02],
         [-4.4050e-02, -8.8507e-02]]),
 tensor([[-0.0131, -0.0203],
         [-0.0263, -0.0406]]))

In [66]:
attention.in_proj_weight.grad[0][0].item()

-5.595298716798425e-05

### Manual

In [57]:
# Input
x_seq_0_pos_0 = tensor(0.2)
x_seq_0_pos_1 = tensor(0.7)

x_seq_1_pos_0 = tensor(0.4)
x_seq_1_pos_1 = tensor(0.5)

# Param

## Q
W_Q_0_0 = tensor(0.1)
W_Q_0_1 = tensor(0.2)

W_Q_1_0 = tensor(0.3)
W_Q_1_1 = tensor(0.4)

## K
W_K_0_0 = tensor(0.2)
W_K_0_1 = tensor(0.9)

W_K_1_0 = tensor(0.4)
W_K_1_1 = tensor(0.5)

## V
W_V_0_0 = tensor(0.3)
W_V_0_1 = tensor(0.4)

W_V_1_0 = tensor(0.5)
W_V_1_1 = tensor(0.6)

## O
W_O_0_0 = tensor(1.0)
W_O_0_1 = tensor(1.1)

W_O_1_0 = tensor(1.2)
W_O_1_1 = tensor(1.3)

## Linear
w0 = tensor(0.1)
w1 = tensor(0.2)

# Forward
Q_0_0 = x_seq_0_pos_0 * W_Q_0_0 + x_seq_0_pos_1 * W_Q_0_1
Q_0_1 = x_seq_0_pos_0 * W_Q_1_0 + x_seq_0_pos_1 * W_Q_1_1

Q_1_0 = x_seq_1_pos_0 * W_Q_0_0 + x_seq_1_pos_1 * W_Q_0_1
Q_1_1 = x_seq_1_pos_0 * W_Q_1_0 + x_seq_1_pos_1 * W_Q_1_1

K_0_0 = x_seq_0_pos_0 * W_K_0_0 + x_seq_0_pos_1 * W_K_0_1
K_0_1 = x_seq_0_pos_0 * W_K_1_0 + x_seq_0_pos_1 * W_K_1_1

K_1_0 = x_seq_1_pos_0 * W_K_0_0 + x_seq_1_pos_1 * W_K_0_1
K_1_1 = x_seq_1_pos_0 * W_K_1_0 + x_seq_1_pos_1 * W_K_1_1

V_0_0 = x_seq_0_pos_0 * W_V_0_0 + x_seq_0_pos_1 * W_V_0_1
V_0_1 = x_seq_0_pos_0 * W_V_1_0 + x_seq_0_pos_1 * W_V_1_1

V_1_0 = x_seq_1_pos_0 * W_V_0_0 + x_seq_1_pos_1 * W_V_0_1
V_1_1 = x_seq_1_pos_0 * W_V_1_0 + x_seq_1_pos_1 * W_V_1_1

score_0_0 = (Q_0_0 * K_0_0 + Q_0_1 * K_0_1) / torch.sqrt(torch.tensor(2.0))
score_0_1 = (Q_0_0 * K_1_0 + Q_0_1 * K_1_1) / torch.sqrt(torch.tensor(2.0))
score_1_0 = (Q_1_0 * K_0_0 + Q_1_1 * K_0_1) / torch.sqrt(torch.tensor(2.0))
score_1_1 = (Q_1_0 * K_1_0 + Q_1_1 * K_1_1) / torch.sqrt(torch.tensor(2.0))

weights_0_0 = torch.exp(score_0_0) / (torch.exp(score_0_0) + torch.exp(score_0_1))
weights_0_1 = torch.exp(score_0_1) / (torch.exp(score_0_0) + torch.exp(score_0_1))

weights_1_0 = torch.exp(score_1_0) / (torch.exp(score_1_0) + torch.exp(score_1_1))
weights_1_1 = torch.exp(score_1_1) / (torch.exp(score_1_0) + torch.exp(score_1_1))

attn_0_0 = weights_0_0 * V_0_0 + weights_0_1 * V_1_0
attn_0_1 = weights_0_0 * V_0_1 + weights_0_1 * V_1_1
attn_1_0 = weights_1_0 * V_0_0 + weights_1_1 * V_1_0
attn_1_1 = weights_1_0 * V_0_1 + weights_1_1 * V_1_1

output_1_0 = attn_1_0 * W_O_0_0 + attn_1_1 * W_O_0_1
output_1_1 = attn_1_0 * W_O_1_0 + attn_1_1 * W_O_1_1

y_pred = output_1_0 * w0 + output_1_1 * w1
y_pred

tensor(0.3010)

In [58]:
y_target = tensor(0.5)
L = (y_pred.item() - y_target.item()) ** 2

In [59]:
dL_dypred = 2 * (y_pred - y_target)
dypred_dw0 = output_1_0
dypred_dw1 = output_1_1

dL_dw0 = dL_dypred * dypred_dw0
dL_dw1 = dL_dypred * dypred_dw1

dL_dw0, dL_dw1

(tensor(-0.3548), tensor(-0.4216))

In [60]:
dq00_dwq00 = x_seq_0_pos_0
dq10_dwq00 = x_seq_1_pos_0

dscore00_dq00 = K_0_0 / torch.sqrt(torch.tensor(2.0))
dscore01_dq00 = K_1_0 / torch.sqrt(torch.tensor(2.0))

dscore10_dq10 = K_0_0 / torch.sqrt(torch.tensor(2.0))
dscore11_dq10 = K_1_0 / torch.sqrt(torch.tensor(2.0))

dweights00_dscore00 = weights_0_0 * (1 - weights_0_0)
dweights00_dscore01 = -weights_0_0 * weights_0_1

dweights01_dscore00 = -weights_0_1 * weights_0_0
dweights01_dscore01 = weights_0_1 * (1 - weights_0_1)

dweights10_dscore10 = weights_1_0 * (1 - weights_1_0)
dweights10_dscore11 = -weights_1_0 * weights_1_1

dweights11_dscore10 = -weights_1_1 * weights_1_0
dweights11_dscore11 = weights_1_1 * (1 - weights_1_1)

dattn00_dweights00 = V_0_0
dattn00_dweights01 = V_1_0

dattn01_dweights00 = V_0_1
dattn01_dweights01 = V_1_1

dattn10_dweights10 = V_0_0
dattn10_dweights11 = V_1_0

dattn11_dweights10 = V_0_1
dattn11_dweights11 = V_1_1

doutput10_dattn10 = W_O_0_0
doutput10_dattn11 = W_O_0_1

doutput11_dattn10 = W_O_1_0
doutput11_dattn11 = W_O_1_1

dypred_doutput10 = w0
dypred_doutput11 = w1

dL_dypred = 2 * (y_pred - y_target)

In [61]:
dL_dypred = 2 * (y_pred - y_target)

dL_doutput10 = dL_dypred * dypred_doutput10
dL_doutput11 = dL_dypred * dypred_doutput11

dL_dattn10 = (
    dL_doutput10 * doutput10_dattn10
    + dL_doutput11 * doutput11_dattn10
)

dL_dattn11 = (
    dL_doutput10 * doutput10_dattn11
    + dL_doutput11 * doutput11_dattn11
)

dL_dweights10 = dL_dattn10 * V_0_0 + dL_dattn11 * V_0_1
dL_dweights11 = dL_dattn10 * V_1_0 + dL_dattn11 * V_1_1

dL_dscore10 = (
    dL_dweights10 * dweights10_dscore10
    + dL_dweights11 * dweights11_dscore10
)

dL_dscore11 = (
    dL_dweights10 * dweights10_dscore11
    + dL_dweights11 * dweights11_dscore11
)

dL_dq10 = (
    dL_dscore10 * dscore10_dq10
    + dL_dscore11 * dscore11_dq10
)

dL_dwq00 = dL_dq10 * dq10_dwq00
dL_dwq00

tensor(-5.5954e-05)

In [62]:
dL_dwq00.item()

-5.5954002164071426e-05